# Assignment 2a — Exploratory Data Analysis (template)

Fill in the `TODO` cells. The full prose, explanations, and worked examples are in the guide:

**[Guide: Exploratory Data Analysis](https://osu-car-msl.github.io/lab-setup-guide/assignments/exploratory-data-analysis/)**

When you finish, publish the notebook as a blog post — see the [Publishing Guide](https://osu-car-msl.github.io/lab-setup-guide/assignments/publishing-guide/).


## Part 0 — Setup

Most setup (repo, venv, directory layout) happens before opening this notebook. See Part 0 of the guide.

Verify your imports work:


In [ ]:
import polars as pl
import altair as alt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Altair can't handle very large datasets in the default renderer
alt.data_transformers.enable("default", max_rows=None)


## Part 1 — Get & Explore the Data

Download the [HCRL Survival IDS dataset](https://ocslab.hksecurity.net/Datasets/survival-ids) into `../data/`.


In [ ]:
# TODO: Load your CSV with polars
# df = pl.read_csv("../data/survival_ids.csv")
df = ...
df.head()


In [ ]:
# TODO: Inspect the schema
df.schema


In [ ]:
# TODO: Summary statistics
df.describe()


In [ ]:
# TODO: Null counts per column
df.null_count()


### Checkpoint

- [ ] Data loaded successfully with `pl.read_csv`
- [ ] Schema reviewed — you understand the column types
- [ ] `df.describe()` output reviewed — you can identify reasonable ranges
- [ ] Missing values checked


## Part 2 — Visualizations

Create at least 3 EDA plots with altair. Save each to `../figures/`.


### 2.1 Distribution plot

Pick a numeric column. Use a binned `mark_bar` with `alt.X(..., bin=alt.Bin(maxbins=50))`.


In [ ]:
# TODO: histogram of a numeric column — save to ../figures/distribution.png
hist = (
    alt.Chart(df)
    .mark_bar()
    .encode(
        # alt.X("<column>:Q", bin=alt.Bin(maxbins=50), title="Value"),
        # alt.Y("count():Q", title="Count"),
    )
    .properties(title="Distribution of ...", width=600, height=350)
)
# hist.save("../figures/distribution.png", scale_factor=2)
hist


### 2.2 Correlation heatmap

Compute correlations on the numeric columns, reshape to long form, then layer `mark_rect` + `mark_text`. Guide: Part 2.2 has the exact recipe.


In [ ]:
# TODO: compute correlation matrix and reshape to long form
numeric = df.select(pl.col(pl.NUMERIC_DTYPES))
corr = numeric.corr()

corr_long = (
    corr.with_columns(pl.Series("row", numeric.columns))
    .unpivot(index="row", variable_name="col", value_name="corr")
)
corr_long.head()


In [ ]:
# TODO: build the heatmap chart — rect layer + text annotation layer
heatmap = (
    alt.Chart(corr_long)
    .mark_rect()
    .encode(
        # alt.X("col:N", sort=numeric.columns),
        # alt.Y("row:N", sort=numeric.columns),
        # alt.Color("corr:Q", scale=alt.Scale(scheme="redblue", domain=[-1, 1])),
    )
    .properties(title="Feature Correlation Heatmap", width=500, height=500)
)
# TODO: add a text layer with alt.condition for light/dark cells
# heatmap.save("../figures/correlation_heatmap.png", scale_factor=2)
heatmap


### 2.3 Class distribution


In [ ]:
# TODO: bar chart of label counts
# class_counts = df.group_by("label_column").len().rename({"len": "count"})
class_counts = ...
bar = (
    alt.Chart(class_counts)
    .mark_bar()
    # .encode(...)
    .properties(title="Class Distribution", width=500, height=300)
)
# bar.save("../figures/class_distribution.png", scale_factor=2)
bar


### Checkpoint

- [ ] Distribution plot created and saved
- [ ] Correlation heatmap created and saved (with value annotations)
- [ ] Class distribution (or third plot) created and saved


## Part 3 — Toy ML Models


In [ ]:
# TODO: prepare features and labels
# X = df.drop("label_column").select(pl.col(pl.NUMERIC_DTYPES)).to_numpy()
# y = df["label_column"].to_numpy()
X, y = ..., ...

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")


In [ ]:
# TODO: train two models
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)


In [ ]:
def confusion_chart(y_true, y_pred, title: str) -> alt.Chart:
    """Build an altair confusion matrix with value annotations."""
    cm = confusion_matrix(y_true, y_pred)
    labels = sorted(set(y_true))
    cm_long = pl.DataFrame({
        "actual": [labels[i] for i in range(len(labels)) for _ in labels],
        "predicted": [labels[j] for _ in labels for j in range(len(labels))],
        "count": cm.flatten().tolist(),
    })
    base = alt.Chart(cm_long).encode(
        alt.X("predicted:N", title="Predicted"),
        alt.Y("actual:N", title="Actual"),
    )
    rects = base.mark_rect().encode(
        alt.Color("count:Q", scale=alt.Scale(scheme="blues"), title="Count"),
        tooltip=["actual:N", "predicted:N", "count:Q"],
    )
    text = base.mark_text(fontSize=14, fontWeight="bold").encode(
        text="count:Q",
        color=alt.condition("datum.count > 50", alt.value("white"), alt.value("black")),
    )
    return (rects + text).properties(title=title, width=300, height=300)


for name, model in [("Logistic Regression", lr), ("Random Forest", rf)]:
    y_pred = model.predict(X_test)
    print(f"\n{'='*40}\n{name}\n{'='*40}")
    print(classification_report(y_test, y_pred))
    chart = confusion_chart(y_test, y_pred, f"Confusion Matrix — {name}")
    # chart.save(f"../figures/confusion_matrix_{name.lower().replace(' ', '_')}.png", scale_factor=2)
    chart.display()


### Checkpoint

- [ ] Train/test split created
- [ ] Two models trained (Logistic Regression + Random Forest)
- [ ] `classification_report` printed for both models
- [ ] Confusion matrix charts created and saved


## Part 4 — Gallery Picks

Browse the [Vega-Altair Example Gallery](https://altair-viz.github.io/gallery/index.html) and pick **2 chart types** that look interesting. Adapt the gallery code to use columns from your dataset.


In [ ]:
# TODO: gallery pick #1 — pick a chart type you haven't used before
chart1 = ...
# chart1.save("../figures/gallery_1.png", scale_factor=2)


In [ ]:
# TODO: gallery pick #2 — different chart type from #1
chart2 = ...
# chart2.save("../figures/gallery_2.png", scale_factor=2)


### Checkpoint

- [ ] Gallery plot 1 created, customized, and saved
- [ ] Gallery plot 2 created, customized, and saved


## Part 5 — Publish

Convert this notebook to a blog post on your Quarto site.

- **Suggested categories:** `[eda, python, visualization, machine-learning]`
- Add YAML frontmatter (see the [Publishing Guide](https://osu-car-msl.github.io/lab-setup-guide/assignments/publishing-guide/))
- Restart kernel + run all cells top-to-bottom before publishing
- Lead the post with your correlation heatmap or best gallery pick

### Final deliverables

- [ ] GitHub repo URL for your EDA project
- [ ] 5+ visualizations in `figures/` (3 EDA + 2 gallery picks)
- [ ] Confusion matrix charts for both models
- [ ] Blog post live on your Quarto site
- [ ] `requirements.txt` in the repo root
